# Nessa distillation LoRA
Trains a small local model on what the cloud teachers (GLM-5.3, Kimi-K3, DeepSeek) did during your real NESSA sessions.

1. On the laptop: `python -m agentharness distill export --out nessa-distill.jsonl`
2. Here: **Runtime > Change runtime type > T4 GPU**, upload `nessa-distill.jsonl`, Run all.
3. Bring the `.gguf` back and `ollama create` it (last cell).

Only turns from runs that ended well (verified / completed / answered) are exported; secrets are scrubbed before export.

In [ ]:
%%capture
!pip install unsloth

In [ ]:
BASE = "unsloth/Qwen2.5-Coder-1.5B-Instruct"   # or -0.5B- (specialist) / -3B-
DATA = "nessa-distill.jsonl"
MAX_LEN = 4096
EPOCHS = 2                                      # small datasets: 2-3; >5k examples: 1
OUT = "nessa-student-1.5b"

In [ ]:
import json
from datasets import Dataset
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained(BASE, max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=0)

def fix_calls(messages):
    # NESSA stores tool-call arguments as JSON strings; chat templates expect objects.
    for m in messages:
        for c in m.get("tool_calls") or []:
            a = c["function"].get("arguments")
            if isinstance(a, str):
                try: c["function"]["arguments"] = json.loads(a or "{}")
                except ValueError: c["function"]["arguments"] = {}
    return messages

rows = [json.loads(l) for l in open(DATA) if l.strip()]
texts = []
for r in rows:
    try:
        texts.append(tok.apply_chat_template(fix_calls(r["messages"]), tools=r.get("tools") or None, tokenize=False))
    except Exception as e:
        print("skip:", e)
texts = [t for t in texts if len(tok(t).input_ids) <= MAX_LEN]
print(len(rows), "examples,", len(texts), "fit in", MAX_LEN, "tokens")
ds = Dataset.from_dict({"text": texts})
split = ds.train_test_split(test_size=max(1, len(ds) // 20), seed=0) if len(ds) >= 40 else {"train": ds, "test": None}
print(split["train"][0]["text"][-1500:])

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(
    model=model, tokenizer=tok, train_dataset=split["train"], eval_dataset=split["test"],
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_LEN, packing=False,
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        num_train_epochs=EPOCHS, learning_rate=1e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
        optim="adamw_8bit", weight_decay=0.01, fp16=True, bf16=False, logging_steps=5,
        eval_strategy="epoch" if split["test"] is not None else "no", save_strategy="no",
        output_dir="out", report_to="none", seed=0))
# Learn only the teacher's replies, not the prompts/tool results it was shown.
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")
trainer.train()

In [ ]:
# sanity check: does the student call a tool like the teacher?
FastLanguageModel.for_inference(model)
r = rows[-1]
p = tok.apply_chat_template(fix_calls(r["messages"][:-1]), tools=r.get("tools") or None, tokenize=False, add_generation_prompt=True)
ids = tok(p, return_tensors="pt").to("cuda")
print("STUDENT:", tok.decode(model.generate(**ids, max_new_tokens=300)[0][ids.input_ids.shape[1]:], skip_special_tokens=False))
print("TEACHER:", r["messages"][-1])

In [ ]:
model.save_pretrained(OUT + "-lora"); tok.save_pretrained(OUT + "-lora")
model.save_pretrained_gguf(OUT, tok, quantization_method="q4_k_m")
!ls -lh {OUT}*/*.gguf *.gguf 2>/dev/null

## Use it in NESSA
```
FROM ./nessa-student-1.5b-Q4_K_M.gguf
PARAMETER num_ctx 32768
PARAMETER temperature 0.2
```
`OLLAMA_HOST=127.0.0.1:11435 ollama create nessa-student -f Modelfile`, then use it as a specialist
(`--subagent-model nessa-student:latest`) or as the main local model (`--model nessa-student:latest`).
Re-export and retrain periodically: the dataset grows as you use NESSA with `--cloud`.